# Tuning & Experiment Tracking

**Week 8 -- CS 203 -- Software Tools and Techniques for AI**

Prof. Nipun Batra, IIT Gandhinagar

---

**Outline:**
1. Grid Search vs Random Search (manual + sklearn)
2. Bayesian Optimization -- 1D GP visualization + `bayesian-optimization`
3. Optuna (TPE) -- pruning, parameter importance
4. GP vs TPE comparison
5. Nested Cross-Validation (the optimism gap)
6. DIY AutoML (pure sklearn -- no broken deps!)
7. PyTorch Reproducibility
8. Experiment Tracking with Trackio

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install numpy matplotlib scikit-learn optuna bayesian-optimization torch trackio

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_digits
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    RandomizedSearchCV, StratifiedKFold
)
from sklearn.metrics import accuracy_score
from scipy.stats import randint, uniform
import time
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

In [ ]:
# ---- Synthetic dataset (used throughout) ----
X, y = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Dataset : {X.shape[0]} samples, {X.shape[1]} features")
print(f"Train   : {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print(f"Class balance: {y.mean():.1%} positive")

---
## Part 1: Grid Search vs Random Search

### 1a. Manual Grid Search (the for-loop way)

Before using sklearn's `GridSearchCV`, let's see what it does under the hood.

In [ ]:
# Manual grid search with nested for loops
from itertools import product

n_estimators_list = [50, 100, 200]
max_depth_list = [5, 10, 15, None]
min_samples_leaf_list = [1, 2, 5]

best_score = -1
best_params = None
all_results = []

total = len(n_estimators_list) * len(max_depth_list) * len(min_samples_leaf_list)
print(f"Total combinations: {total}")
print(f"With 5-fold CV, that's {total * 5} model fits!\n")

t0 = time.time()
for n_est, depth, leaf in product(n_estimators_list, max_depth_list, min_samples_leaf_list):
    model = RandomForestClassifier(
        n_estimators=n_est, max_depth=depth,
        min_samples_leaf=leaf, random_state=42
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    mean_score = scores.mean()
    all_results.append({
        'n_estimators': n_est, 'max_depth': depth,
        'min_samples_leaf': leaf, 'score': mean_score
    })
    if mean_score > best_score:
        best_score = mean_score
        best_params = {'n_estimators': n_est, 'max_depth': depth, 'min_samples_leaf': leaf}

elapsed = time.time() - t0
print(f"Manual Grid Search ({elapsed:.1f}s)")
print(f"Best CV score : {best_score:.4f}")
print(f"Best params   : {best_params}")
print(f"\nProblem: adding one more hyperparameter multiplies the combos!")
print(f"  3 x 4 x 3 = {total}  -->  3 x 4 x 3 x 5 = {total * 5} (curse of dimensionality)")

### 1b. sklearn `GridSearchCV` (the clean way)

In [ ]:
# --- Grid Search with sklearn ---
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [1, 2, 5],
}

grid_cv = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True,
)
grid_cv.fit(X_train, y_train)

BUDGET = len(grid_cv.cv_results_['params'])
print(f"Grid Search: {BUDGET} combinations evaluated")
print(f"Best CV score : {grid_cv.best_score_:.4f}")
print(f"Best params   : {grid_cv.best_params_}")

### 1c. Random Search

In [ ]:
# --- Random Search (same budget as grid) ---
param_distributions = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(3, 30),
    'min_samples_leaf': randint(1, 20),
    'max_features': uniform(0.1, 0.9),
}

random_cv = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions,
    n_iter=BUDGET,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
)
random_cv.fit(X_train, y_train)

print(f"Random Search: {BUDGET} combinations evaluated")
print(f"Best CV score : {random_cv.best_score_:.4f}")
print(f"Best params   : {random_cv.best_params_}")
print()
diff = random_cv.best_score_ - grid_cv.best_score_
winner = "Random" if diff > 0 else "Grid"
print(f"{winner} Search wins by {abs(diff):.4f}")

In [ ]:
# --- Visualize: Grid covers a lattice, Random covers the space ---
grid_params = grid_cv.cv_results_['params']
rand_params = random_cv.cv_results_['params']

def extract(params_list, key, default=None):
    vals = []
    for p in params_list:
        v = p.get(key, default)
        vals.append(v if v is not None else 30)
    return np.array(vals, dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, params, scores, title in [
    (axes[0], grid_params, grid_cv.cv_results_['mean_test_score'], f'Grid Search ({BUDGET} evals)'),
    (axes[1], rand_params, random_cv.cv_results_['mean_test_score'], f'Random Search ({BUDGET} evals)'),
]:
    ne = extract(params, 'n_estimators')
    md = extract(params, 'max_depth')
    sc = ax.scatter(ne, md, c=scores, cmap='viridis', s=60, edgecolors='k', linewidths=0.5)
    ax.set_xlabel('n_estimators')
    ax.set_ylabel('max_depth')
    ax.set_title(title)
    plt.colorbar(sc, ax=ax, label='CV Accuracy')

fig.suptitle('Grid covers a lattice | Random covers the space uniformly', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Key insight (Bergstra & Bengio, JMLR 2012):** When only a few hyperparameters matter (which is typical), random search finds good values faster because it samples more unique values per important dimension.

---
## Part 2: Bayesian Optimization

### 2a. 1D Visualization -- How GP-based Bayesian Optimization Works

Instead of searching blindly, Bayesian optimization:
1. Fits a **surrogate model** (Gaussian Process) to observed points
2. Uses an **acquisition function** to decide where to sample next
3. Balances **exploitation** (near known good points) and **exploration** (uncertain regions)

In [ ]:
from bayes_opt import BayesianOptimization, UtilityFunction

# A 1D function we want to maximize (optimizer doesn't see the formula)
def target_function(x):
    return np.sin(x) * x + np.cos(2 * x)

x_range = np.linspace(0, 6, 300)
y_true = [target_function(x) for x in x_range]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_range, y_true, 'b-', linewidth=2, label='True function (unknown to optimizer)')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('Goal: find the maximum of this function', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Run Bayesian Optimization step by step
optimizer = BayesianOptimization(
    f=target_function,
    pbounds={'x': (0, 6)},
    random_state=42,
    verbose=0
)

# 3 random initialization points
optimizer.maximize(init_points=3, n_iter=0)
print("Initial random points:")
for i, res in enumerate(optimizer.res):
    print(f"  Point {i+1}: x={res['params']['x']:.3f}, f(x)={res['target']:.3f}")

# 7 guided iterations
optimizer.maximize(init_points=0, n_iter=7)
print(f"\nAfter 10 total evaluations:")
print(f"Best: x={optimizer.max['params']['x']:.3f}, f(x)={optimizer.max['target']:.3f}")

In [ ]:
# Visualize the optimization trajectory
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_range, y_true, 'b-', linewidth=2, alpha=0.4, label='True function')

xs = [res['params']['x'] for res in optimizer.res]
ys = [res['target'] for res in optimizer.res]

colors = plt.cm.Reds(np.linspace(0.3, 1.0, len(xs)))
for i, (x, y_val, c) in enumerate(zip(xs, ys, colors)):
    marker = 's' if i < 3 else 'o'  # squares = random, circles = guided
    ax.scatter(x, y_val, c=[c], s=120, edgecolors='k', marker=marker, zorder=5)
    ax.annotate(f'{i+1}', (x, y_val), textcoords='offset points',
                xytext=(5, 10), fontsize=10, fontweight='bold')

best_x = optimizer.max['params']['x']
best_y = optimizer.max['target']
ax.scatter([best_x], [best_y], c='green', s=250, marker='*', zorder=6,
           edgecolors='k', label=f'Best: x={best_x:.2f}, f(x)={best_y:.2f}')

ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('f(x)', fontsize=13)
ax.set_title('Bayesian Optimization: 3 random (squares) + 7 guided (circles)', fontsize=13)
ax.legend(loc='lower left', fontsize=11)
plt.tight_layout()
plt.show()
print("Notice: guided points cluster near the optimum -- the GP learns where to look.")

### 2b. Using `bayesian-optimization` for hyperparameter tuning

Now let's use GP-based BayesOpt to tune a real model.

In [ ]:
# GP-based Bayesian optimization for RandomForest hyperparameters
def rf_objective(n_estimators, max_depth, min_samples_leaf):
    """Objective function for bayesian-optimization (must return a scalar to maximize)."""
    model = RandomForestClassifier(
        n_estimators=int(n_estimators),
        max_depth=int(max_depth),
        min_samples_leaf=int(min_samples_leaf),
        random_state=42
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return scores.mean()

bo_optimizer = BayesianOptimization(
    f=rf_objective,
    pbounds={
        'n_estimators': (50, 500),
        'max_depth': (3, 30),
        'min_samples_leaf': (1, 20),
    },
    random_state=42,
    verbose=0
)

bo_optimizer.maximize(init_points=5, n_iter=BUDGET - 5)

print(f"GP-BayesOpt: {BUDGET} evaluations")
print(f"Best CV score : {bo_optimizer.max['target']:.4f}")
bp = bo_optimizer.max['params']
print(f"Best params   : n_estimators={int(bp['n_estimators'])}, "
      f"max_depth={int(bp['max_depth'])}, min_samples_leaf={int(bp['min_samples_leaf'])}")

### 2c. Optuna (TPE-based Bayesian Optimization)

Optuna uses **Tree Parzen Estimators (TPE)** instead of Gaussian Processes.
- TPE scales better to many parameters
- Handles categorical and conditional parameters naturally
- Supports **pruning** (early stopping of bad trials)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

OPTUNA_BUDGET = 100

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_float('max_features', 0.1, 1.0),
    }
    model = RandomForestClassifier(**params, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=OPTUNA_BUDGET)

print(f"Optuna (TPE): {OPTUNA_BUDGET} trials")
print(f"Best CV score : {study.best_value:.4f}")
print(f"Best params   : {study.best_params}")

In [ ]:
# Optuna optimization history: how score improves over trials
trial_values = [t.value for t in study.trials]
best_so_far = np.maximum.accumulate(trial_values)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(range(len(trial_values)), trial_values, alpha=0.3, s=20, label='Individual trials')
ax.plot(best_so_far, 'r-', linewidth=2, label='Best so far (Optuna)')
ax.axhline(grid_cv.best_score_, color='blue', linestyle='--', alpha=0.7,
           label=f'Grid best ({grid_cv.best_score_:.4f})')
ax.axhline(random_cv.best_score_, color='green', linestyle='--', alpha=0.7,
           label=f'Random best ({random_cv.best_score_:.4f})')
ax.set_xlabel('Trial number', fontsize=12)
ax.set_ylabel('CV Accuracy', fontsize=12)
ax.set_title('Optuna Optimization History: TPE learns over time', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Optuna: which hyperparameters matter most?
fig = optuna.visualization.plot_param_importances(study)
fig.show()

In [ ]:
# Optuna: contour plot -- how pairs of params interact
fig = optuna.visualization.plot_contour(study, params=['max_depth', 'min_samples_leaf'])
fig.show()

### 2d. Optuna with Pruning (early stopping bad trials)

One of Optuna's killer features: it can **prune** (stop early) unpromising trials during cross-validation. This saves a lot of time.

In [ ]:
# Optuna with pruning -- report intermediate CV fold scores
def objective_with_pruning(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_float('max_features', 0.1, 1.0),
    }
    model = RandomForestClassifier(**params, random_state=42)
    
    # Report each fold as intermediate value -- Optuna can prune early
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
        model.fit(X_train[train_idx], y_train[train_idx])
        score = model.score(X_train[val_idx], y_train[val_idx])
        fold_scores.append(score)
        trial.report(np.mean(fold_scores), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)

pruned_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)
)
pruned_study.optimize(objective_with_pruning, n_trials=100)

n_pruned = len([t for t in pruned_study.trials if t.state == optuna.trial.TrialState.PRUNED])
n_complete = len([t for t in pruned_study.trials if t.state == optuna.trial.TrialState.COMPLETE])

print(f"Pruned study: {n_complete} completed, {n_pruned} pruned (saved {n_pruned * 5} fold evaluations!)")
print(f"Best CV score: {pruned_study.best_value:.4f}")
print(f"Best params  : {pruned_study.best_params}")

### 2e. Compare all tuning methods

In [ ]:
# Summary comparison
methods = {
    'Grid Search': (BUDGET, grid_cv.best_score_),
    'Random Search': (BUDGET, random_cv.best_score_),
    'GP-BayesOpt': (BUDGET, bo_optimizer.max['target']),
    'Optuna (TPE)': (OPTUNA_BUDGET, study.best_value),
    'Optuna+Pruning': (100, pruned_study.best_value),
}

print(f"{'Method':<20} {'Budget':>8} {'Best CV Acc':>12}")
print("=" * 42)
for name, (budget, score) in methods.items():
    print(f"{name:<20} {budget:>8} {score:>12.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
names = list(methods.keys())
scores = [v[1] for v in methods.values()]
budgets = [v[0] for v in methods.values()]
colors = ['#4C72B0', '#55A868', '#DD8452', '#C44E52', '#8172B3']
bars = ax.bar(names, scores, color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Best CV Accuracy', fontsize=12)
ax.set_title('Tuning Methods Comparison', fontsize=13)
ax.set_ylim(min(scores) - 0.015, max(scores) + 0.008)
for bar, score, budget in zip(bars, scores, budgets):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f'{score:.4f}\n({budget} evals)', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: GP vs TPE -- When to Use Which?

| | GP (Gaussian Process) | TPE (Tree Parzen Estimator) |
|---|---|---|
| **Surrogate** | Global function model | Density ratios |
| **Scales to** | ~20 params | 100s of params |
| **Categorical** | Needs encoding | Native support |
| **Conditional** | Hard | Easy |
| **Library** | `bayesian-optimization` | `optuna` |
| **Best for** | Small search spaces, continuous | Large, mixed search spaces |

---
## Part 4: Nested Cross-Validation

`GridSearchCV.best_score_` is **optimistic** -- it reports the best score found after searching. This is selection bias. Nested CV gives an unbiased estimate of how well your tuned model will generalize.

In [ ]:
# Inner CV: tunes hyperparameters
inner_cv = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={'max_depth': [5, 10, 15, None], 'n_estimators': [50, 100, 200]},
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
)

# Outer CV: evaluates the full tune-then-predict pipeline
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_scores = cross_val_score(inner_cv, X, y, cv=outer_cv, scoring='accuracy')

# Also get the (optimistic) best_score_ from a single fit
inner_cv.fit(X, y)
optimistic_score = inner_cv.best_score_
nested_score = outer_scores.mean()
gap = optimistic_score - nested_score

print("=" * 60)
print(f"  GridSearchCV.best_score_ (OPTIMISTIC) : {optimistic_score:.4f}")
print(f"  Nested CV outer scores                : {outer_scores}")
print(f"  Nested CV mean (UNBIASED)             : {nested_score:.4f} +/- {outer_scores.std():.4f}")
print(f"  Optimism gap                          : {gap:.4f}")
print("=" * 60)

In [ ]:
# Visualize the gap
fig, ax = plt.subplots(figsize=(8, 3.5))

ax.barh(['Nested CV\n(unbiased)', 'GridSearchCV.best_score_\n(optimistic)'],
        [nested_score, optimistic_score],
        color=['#55A868', '#C44E52'], edgecolor='black', linewidth=0.8, height=0.5)

ax.errorbar(nested_score, 0, xerr=outer_scores.std(), fmt='none',
            ecolor='black', capsize=5, linewidth=2)

ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title(f'The Optimism Gap: {gap:.4f}', fontsize=13)
ax.set_xlim(min(nested_score - outer_scores.std(), optimistic_score) - 0.02,
            max(nested_score, optimistic_score) + 0.02)

for i, v in enumerate([nested_score, optimistic_score]):
    ax.text(v + 0.003, i, f'{v:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Part 5: DIY AutoML (pure sklearn)

AutoML = automatically try multiple model families with tuning. Here we build it ourselves -- no extra dependencies needed.

In [ ]:
# DIY AutoML: try multiple models with their own hyperparameter spaces
model_configs = {
    'Logistic Regression': {
        'model': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
        'params': {
            'clf__C': [0.01, 0.1, 1, 10, 100],
            'clf__penalty': ['l1', 'l2'],
            'clf__solver': ['liblinear'],
        }
    },
    'KNN': {
        'model': Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())]),
        'params': {
            'clf__n_neighbors': [3, 5, 7, 11, 15],
            'clf__weights': ['uniform', 'distance'],
            'clf__metric': ['euclidean', 'manhattan'],
        }
    },
    'SVM': {
        'model': Pipeline([('scaler', StandardScaler()), ('clf', SVC())]),
        'params': {
            'clf__C': [0.1, 1, 10],
            'clf__kernel': ['rbf', 'poly'],
            'clf__gamma': ['scale', 'auto'],
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, 15, None],
            'min_samples_leaf': [1, 2, 5],
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.1, 0.3],
            'max_depth': [3, 5, 7],
        }
    },
    'Extra Trees': {
        'model': ExtraTreesClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, 15, None],
            'min_samples_leaf': [1, 2, 5],
        }
    },
}

automl_results = {}
print(f"{'Model':<25} {'Combos':>8} {'Best CV':>10} {'Time':>8}")
print("=" * 55)

for name, config in model_configs.items():
    t0 = time.time()
    gs = GridSearchCV(
        config['model'], config['params'],
        cv=5, scoring='accuracy', n_jobs=-1
    )
    gs.fit(X_train, y_train)
    elapsed = time.time() - t0
    n_combos = len(gs.cv_results_['params'])
    automl_results[name] = {
        'best_score': gs.best_score_,
        'best_params': gs.best_params_,
        'best_estimator': gs.best_estimator_,
        'n_combos': n_combos,
        'time': elapsed,
    }
    print(f"{name:<25} {n_combos:>8} {gs.best_score_:>10.4f} {elapsed:>7.1f}s")

# Find the winner
winner_name = max(automl_results, key=lambda k: automl_results[k]['best_score'])
winner = automl_results[winner_name]
print(f"\nAutoML winner: {winner_name} (CV={winner['best_score']:.4f})")
print(f"Best params: {winner['best_params']}")

In [ ]:
# Evaluate all models on the held-out test set
fig, ax = plt.subplots(figsize=(10, 5))

names = list(automl_results.keys())
cv_scores = [automl_results[n]['best_score'] for n in names]
test_scores = [automl_results[n]['best_estimator'].score(X_test, y_test) for n in names]

x_pos = np.arange(len(names))
width = 0.35
bars1 = ax.bar(x_pos - width/2, cv_scores, width, label='CV Score', color='#4C72B0', edgecolor='black')
bars2 = ax.bar(x_pos + width/2, test_scores, width, label='Test Score', color='#55A868', edgecolor='black')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('DIY AutoML: CV vs Test Accuracy for 6 Model Families', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0.8, 1.0)

for bar, score in zip(bars1, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{score:.3f}', ha='center', fontsize=8)
for bar, score in zip(bars2, test_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{score:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## Part 6: Optuna for Neural Networks

Optuna really shines when tuning neural networks -- conditional search spaces, pruning, and more.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print(f"PyTorch version: {torch.__version__}")

# Prepare data as tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

In [ ]:
def create_model(trial):
    """Create a neural network with Optuna-suggested architecture."""
    n_layers = trial.suggest_int('n_layers', 1, 3)
    layers = []
    in_features = 10  # our input dimension
    
    for i in range(n_layers):
        out_features = trial.suggest_int(f'n_units_l{i}', 16, 128)
        layers.append(nn.Linear(in_features, out_features))
        
        # Optuna can choose activation function per layer
        activation = trial.suggest_categorical(f'activation_l{i}', ['relu', 'tanh'])
        layers.append(nn.ReLU() if activation == 'relu' else nn.Tanh())
        
        # Optuna can choose dropout per layer
        dropout = trial.suggest_float(f'dropout_l{i}', 0.0, 0.5)
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        
        in_features = out_features
    
    layers.append(nn.Linear(in_features, 2))
    return nn.Sequential(*layers)


def nn_objective(trial):
    """Optuna objective for neural network tuning."""
    model = create_model(trial)
    
    lr = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD'])
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    
    optimizer_cls = optim.Adam if optimizer_name == 'Adam' else optim.SGD
    opt = optimizer_cls(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    dataset = TensorDataset(X_train_t, y_train_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Train for 30 epochs with pruning
    for epoch in range(30):
        model.train()
        for batch_X, batch_y in loader:
            opt.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            opt.step()
        
        # Evaluate and report for pruning
        model.eval()
        with torch.no_grad():
            preds = model(X_test_t).argmax(dim=1)
            accuracy = (preds == y_test_t).float().mean().item()
        
        trial.report(accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return accuracy


nn_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
)
nn_study.optimize(nn_objective, n_trials=50, show_progress_bar=True)

n_pruned = len([t for t in nn_study.trials if t.state == optuna.trial.TrialState.PRUNED])
n_complete = len([t for t in nn_study.trials if t.state == optuna.trial.TrialState.COMPLETE])

print(f"\nNeural Network Tuning: {n_complete} completed, {n_pruned} pruned")
print(f"Best accuracy: {nn_study.best_value:.4f}")
print(f"Best params:")
for k, v in nn_study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Visualize neural network tuning
fig = optuna.visualization.plot_optimization_history(nn_study)
fig.update_layout(title='Neural Network Tuning: Optimization History')
fig.show()

---
## Part 7: PyTorch Reproducibility

In [ ]:
import random
import os

def set_seed_full(seed=42):
    """Complete seed function for full PyTorch reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)

In [ ]:
# WITHOUT seeds: different output every run
print("=== WITHOUT seeds (outputs differ each run) ===")
outputs_no_seed = []
for i in range(3):
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    val = model(x)[0].item()
    outputs_no_seed.append(val)
    print(f"  Run {i+1}: first output = {val:.6f}")
print(f"  All same? {len(set([f'{v:.6f}' for v in outputs_no_seed])) == 1}")

In [ ]:
# WITH set_seed_full: identical output every run
print("=== WITH set_seed_full (outputs are identical) ===")
outputs_seeded = []
for i in range(3):
    set_seed_full(42)
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    val = model(x)[0].item()
    outputs_seeded.append(val)
    print(f"  Run {i+1}: first output = {val:.6f}")
print(f"  All same? {len(set([f'{v:.6f}' for v in outputs_seeded])) == 1}")

In [ ]:
# Multi-seed reporting: honest performance estimation
print("=== Multi-Seed Reporting ===")
seed_results = []
seeds = [42, 123, 456, 789, 1024]

for seed in seeds:
    np.random.seed(seed)
    X_s, y_s = make_classification(
        n_samples=500, n_features=10, n_informative=5,
        n_redundant=2, random_state=seed
    )
    Xtr, Xte, ytr, yte = train_test_split(X_s, y_s, test_size=0.2, random_state=seed)
    m = RandomForestClassifier(n_estimators=100, random_state=seed)
    m.fit(Xtr, ytr)
    acc = m.score(Xte, yte)
    seed_results.append(acc)
    print(f"  Seed {seed:>4d}: accuracy = {acc:.4f}")

mean_acc = np.mean(seed_results)
std_acc = np.std(seed_results)
print(f"\nResult: {mean_acc:.4f} +/- {std_acc:.4f}")
print("This is more honest than reporting a single lucky seed.")

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh([f'Seed {s}' for s in seeds], seed_results, color='#4C72B0',
        edgecolor='black', linewidth=0.5)
ax.axvline(mean_acc, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_acc:.4f}')
ax.set_xlabel('Test Accuracy')
ax.set_title('Multi-Seed Reporting')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 8: Experiment Tracking with Trackio

[Trackio](https://github.com/huggingface/trackio) is a **local-first, free** experiment tracking library from Hugging Face. It stores everything on your machine -- no cloud account needed.

```bash
pip install trackio
```

To view the dashboard after logging experiments:
```bash
trackio
```

In [ ]:
import trackio

### 8a. Basic: Log sklearn model results

In [ ]:
# Run 1: Logistic Regression baseline
trackio.init(project="cs203-tuning-demo", name="logistic-baseline", config={
    "model": "LogisticRegression", "C": 1.0, "max_iter": 1000,
})

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
train_acc = lr_model.score(X_train, y_train)
test_acc = lr_model.score(X_test, y_test)

trackio.log({"train_accuracy": train_acc, "test_accuracy": test_acc, "train_test_gap": train_acc - test_acc})
trackio.finish()
print(f"Logistic Regression -- Train: {train_acc:.4f}, Test: {test_acc:.4f}")

In [ ]:
# Run 2: Tuned Random Forest
trackio.init(project="cs203-tuning-demo", name="rf-grid-tuned", config={
    "model": "RandomForest",
    "tuning": "GridSearchCV",
    **grid_cv.best_params_,
})

rf_model = grid_cv.best_estimator_
train_acc_rf = rf_model.score(X_train, y_train)
test_acc_rf = rf_model.score(X_test, y_test)

trackio.log({
    "train_accuracy": train_acc_rf, "test_accuracy": test_acc_rf,
    "train_test_gap": train_acc_rf - test_acc_rf,
    "cv_score": grid_cv.best_score_,
})
trackio.finish()
print(f"Random Forest (tuned) -- Train: {train_acc_rf:.4f}, Test: {test_acc_rf:.4f}")

In [ ]:
# Run 3: Gradient Boosting (from DIY AutoML results)
gb_result = automl_results['Gradient Boosting']
trackio.init(project="cs203-tuning-demo", name="gb-tuned", config={
    "model": "GradientBoosting",
    "tuning": "GridSearchCV",
    **gb_result['best_params'],
})

gb_model = gb_result['best_estimator']
train_acc_gb = gb_model.score(X_train, y_train)
test_acc_gb = gb_model.score(X_test, y_test)

trackio.log({
    "train_accuracy": train_acc_gb, "test_accuracy": test_acc_gb,
    "train_test_gap": train_acc_gb - test_acc_gb,
    "cv_score": gb_result['best_score'],
})
trackio.finish()
print(f"Gradient Boosting (tuned) -- Train: {train_acc_gb:.4f}, Test: {test_acc_gb:.4f}")

### 8b. Training Curves: Log metrics over epochs

Trackio really shines when you log metrics step-by-step during training.

In [ ]:
# Log Random Forest performance as we increase n_estimators
trackio.init(project="cs203-tuning-demo", name="rf-n-estimators-sweep", config={
    "model": "RandomForest", "experiment": "n_estimators_sweep",
    "max_depth": 10, "min_samples_leaf": 1,
})

for n_trees in range(10, 310, 10):
    rf = RandomForestClassifier(n_estimators=n_trees, max_depth=10,
                                min_samples_leaf=1, random_state=42)
    rf.fit(X_train, y_train)
    trackio.log({
        "n_estimators": n_trees,
        "train_accuracy": rf.score(X_train, y_train),
        "test_accuracy": rf.score(X_test, y_test),
    })

trackio.finish()
print("Logged 30 steps of n_estimators sweep")

### 8c. Neural Network Training with Trackio

Track a full PyTorch training loop -- loss, accuracy, learning rate -- all visible in the dashboard.

In [ ]:
# Train a neural network and log everything to Trackio
set_seed_full(42)

# Build a simple NN
net = nn.Sequential(
    nn.Linear(10, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(32, 2)
)

criterion = nn.CrossEntropyLoss()
optimizer_nn = optim.Adam(net.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer_nn, step_size=20, gamma=0.5)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize Trackio with full config
trackio.init(project="cs203-tuning-demo", name="nn-training-run", config={
    "model": "MLP",
    "architecture": "10->64->32->2",
    "optimizer": "Adam",
    "initial_lr": 0.001,
    "scheduler": "StepLR(step=20, gamma=0.5)",
    "batch_size": 64,
    "epochs": 60,
    "dropout": 0.2,
})

for epoch in range(60):
    # Train
    net.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer_nn.zero_grad()
        loss = criterion(net(batch_X), batch_y)
        loss.backward()
        optimizer_nn.step()
        epoch_loss += loss.item()
    
    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    
    # Evaluate
    net.eval()
    with torch.no_grad():
        train_preds = net(X_train_t).argmax(dim=1)
        test_preds = net(X_test_t).argmax(dim=1)
        train_acc_nn = (train_preds == y_train_t).float().mean().item()
        test_acc_nn = (test_preds == y_test_t).float().mean().item()
    
    # Log to Trackio
    trackio.log({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        "train_accuracy": train_acc_nn,
        "test_accuracy": test_acc_nn,
        "learning_rate": scheduler.get_last_lr()[0],
        "train_test_gap": train_acc_nn - test_acc_nn,
    })
    
    if (epoch + 1) % 15 == 0:
        print(f"Epoch {epoch+1:3d}: loss={avg_loss:.4f}, "
              f"train_acc={train_acc_nn:.4f}, test_acc={test_acc_nn:.4f}, "
              f"lr={scheduler.get_last_lr()[0]:.6f}")

trackio.finish()
print(f"\nFinal: train={train_acc_nn:.4f}, test={test_acc_nn:.4f}")

### 8d. Compare multiple learning rates

Track several experiments with different hyperparameters and compare them in the dashboard.

In [ ]:
# Train with different learning rates and compare
learning_rates = [0.01, 0.001, 0.0001]
lr_histories = {}

for lr_val in learning_rates:
    set_seed_full(42)
    net = nn.Sequential(
        nn.Linear(10, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 2)
    )
    opt_lr = optim.Adam(net.parameters(), lr=lr_val)
    criterion = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
    
    trackio.init(project="cs203-tuning-demo", name=f"nn-lr-{lr_val}", config={
        "model": "MLP", "lr": lr_val, "experiment": "lr_comparison",
    })
    
    test_accs = []
    for epoch in range(40):
        net.train()
        epoch_loss = 0
        for bx, by in loader:
            opt_lr.zero_grad()
            loss = criterion(net(bx), by)
            loss.backward()
            opt_lr.step()
            epoch_loss += loss.item()
        
        net.eval()
        with torch.no_grad():
            test_acc_val = (net(X_test_t).argmax(1) == y_test_t).float().mean().item()
        test_accs.append(test_acc_val)
        
        trackio.log({
            "epoch": epoch + 1,
            "train_loss": epoch_loss / len(loader),
            "test_accuracy": test_acc_val,
        })
    
    trackio.finish()
    lr_histories[lr_val] = test_accs
    print(f"lr={lr_val:.4f}: final test_acc={test_accs[-1]:.4f}")

# Plot locally too
fig, ax = plt.subplots(figsize=(10, 5))
for lr_val, accs in lr_histories.items():
    ax.plot(range(1, 41), accs, label=f'lr={lr_val}', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Learning Rate Comparison (also visible in Trackio dashboard)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 65)
print("All experiments are logged! To view the dashboard:")
print()
print("  $ trackio")
print()
print("This opens a local Gradio dashboard where you can:")
print("  - Compare all runs side-by-side")
print("  - View training curves (loss, accuracy, lr)")
print("  - Filter runs by config (model, lr, experiment)")
print("  - See system metrics (CPU, memory on Apple Silicon)")
print("  - Export results to CSV")
print()
print("Data stored locally: ~/.cache/huggingface/trackio/")
print("No cloud account, no signup, no API keys!")
print("=" * 65)

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| Grid Search | Exhaustive but scales poorly (curse of dimensionality) |
| Random Search | Covers space better; same budget, often better results |
| GP-BayesOpt | Models the function with a GP; great for <20 continuous params |
| Optuna (TPE) | Scales to many params, handles categorical, supports pruning |
| Nested CV | `best_score_` is optimistic; nested CV gives unbiased estimate |
| DIY AutoML | Try multiple model families automatically (no extra deps) |
| Reproducibility | `set_seed_full()` + multi-seed reporting for honest results |
| Trackio | Local-first, free experiment tracking with Gradio dashboard |